# Programming Project #1: Hybrid Images

## CS445: Computational Photography

In [1]:
import cv2

import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
%matplotlib widget

import math

# My local directory
datadir = "/Users/garimajajoo/Desktop/CS445/final/" 

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

def prompt_selection(image,num_clicks,title):
    fig = plt.figure(figsize=(15,10))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

    points = plt.ginput(num_clicks)

    plt.close(fig)
    clear_output(wait=True)

    clicked = np.array(points, dtype=np.float32)
    return clicked

In [4]:
im_file = datadir + 'bell_tower.png'
im = np.float32(cv2.imread(im_file, cv2.COLOR_BGR2RGB) / 255.0)

In [5]:
def calculate_heights(vanishing_line, ref_object, ref_height, target_points):
    '''
    Inputs:
        vanishing_line: 2 x 2 numpy array that represents 2 points on the vanishing line
        ref_object: 2x2 numpy array that represents the bottom point and top point of a reference object
        ref_height: int representation of the height of reference object
        target_points: 2n x 2 numpy array that represents the bottom and top points for n objects.
        The bottom point of an object is always followed by the top point.
        
    Output:
        Returns a list of heights. The order is respective to the order of the target points.
    '''  
    num_points=target_points.shape[0]
    assert(num_points%2==0)
    
    # appending the 1 for homogeneous coordinates
    vx=np.array([vanishing_line[0,0], vanishing_line[0,1], 1])
    vy=np.array([vanishing_line[1,0], vanishing_line[1,1], 1])
    b_ref=np.array([ref_object[0,0],ref_object[0,1],1])
    t_ref=np.array([ref_object[1,0],ref_object[1,1],1])
    homogeneous_targets = np.zeros([num_points,3])
    for i in range(num_points):
        homogeneous_targets[i]=np.array([target_points[i,0], target_points[i,1],1])

    height_list=[]
    for i in range(0,num_points,2):
        b_tar = np.array([homogeneous_targets[i,0],homogeneous_targets[i,1],1])
        t_tar = np.array([homogeneous_targets[i+1,0],homogeneous_targets[i+1,1],1])
        
        # horizon
        horizon = np.cross(vx, vy)
        
        # vanishing point along ground direction between objects
        v = np.cross(np.cross(b_ref, b_tar), horizon)
        v = v / v[2]
        
        t_transfer = np.cross(np.cross(v, t_ref), np.cross(t_tar,b_tar))
        t_transfer = t_transfer / t_transfer[2]
        
        # height ratio
        height = ref_height * abs(b_tar[1] - t_tar[1]) / abs(b_tar[1] - t_transfer[1]) 
        height_list.append(height)

    return height_list
    

In [7]:
vps = prompt_selection(im, 2, 'Click on 2 points on the vanishing line')
ref = prompt_selection(im, 2, 'Click on the bottom point of an object with a known height. Then click on the top of that object.')
targets = prompt_selection(im, -1, 'Click on the bottom and top points of as many objects. When you are done selecting objects, press Enter')
ref_height = 56
calculate_heights(vps,ref,ref_height,targets)

[np.float64(12.538237729731872),
 np.float64(23.39444832581794),
 np.float64(19.540902307774815)]